# Chess Position Evaluation with Graph Neural Networks

A ResGATv2 model that evaluates chess positions by representing the board as a graph:
- **64 nodes** (one per square) with 21-dimensional features (piece one-hot, position, castling rights, etc.)
- **Edges** from legal moves with 6-dimensional features (capture, promotion, displacement, etc.)
- **WDL classification** (Win/Draw/Loss) instead of raw centipawn regression for stable training

Based on research from AlphaGateau (NeurIPS 2024), Alwer & Plaat (2023), and Czech et al. (2023).

In [ ]:
!pip install -q torch torch_geometric chess pandas numpy scikit-learn matplotlib tqdm

In [ ]:
# ============================================================
# Configuration — edit these to control the run
# ============================================================

# Path to CSV dataset (Kaggle default)
DATA_PATH = "/kaggle/input/stockfish-best-moves-compilation/dataset_eval.csv"

# How many positions to use (None = full dataset)
NUM_SAMPLES = 50_000

# Training hyperparameters
BATCH_SIZE = 256
EPOCHS = 200
LR = 1e-3
PATIENCE = 20
WARMUP_EPOCHS = 5

# Model architecture
HIDDEN = 128
HEADS = 4
NUM_BLOCKS = 4
DROPOUT = 0.2

# Caching (saves processed graphs to disk so reruns are fast)
USE_CACHE = True
CACHE_PATH = f"processed_graphs_{NUM_SAMPLES}.pt"

# Output paths
BEST_MODEL_PATH = "best_model.pt"

print("Configuration loaded.")

In [ ]:
# ============================================================
# Imports and device setup
# ============================================================

import re
import math
import time
import logging
from pathlib import Path

import chess
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv, global_mean_pool, BatchNorm
from tqdm.auto import tqdm

# ── Logging ──
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("ChessGCN")

# ── Device ──
if torch.cuda.is_available():
    device = torch.device("cuda")
    log.info(f"Using CUDA — {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    log.info("Using MPS (Apple Silicon)")
else:
    device = torch.device("cpu")
    log.info("Using CPU")

log.info(f"PyTorch {torch.__version__}")

## 1. Data Pipeline

**Evaluation parsing** → **WDL conversion** → **FEN to graph** → **Stratified sampling** → **Caching**

In [ ]:
# ============================================================
# Data pipeline: FEN → Graph with WDL labels
# ============================================================

# Stockfish WDL calibration constant
WDL_K = 111.0


def parse_evaluation(eval_str):
    """Convert evaluation string to centipawns. Mate → ±10000 scaled by distance."""
    mate_match = re.match(r"M(-?\d+)", str(eval_str))
    if mate_match:
        mate_moves = int(mate_match.group(1))
        if mate_moves > 0:
            return 10000 - (mate_moves - 1) * 10
        else:
            return -10000 + (abs(mate_moves) - 1) * 10
    return float(eval_str)


def cp_to_wdl(cp):
    """Convert centipawn evaluation to (win, draw, loss) probabilities."""
    if cp >= 9000:
        return (1.0, 0.0, 0.0)
    if cp <= -9000:
        return (0.0, 0.0, 1.0)
    win = 1.0 / (1.0 + math.exp(-cp / WDL_K))
    loss = 1.0 / (1.0 + math.exp(cp / WDL_K))
    draw = max(1.0 - win - loss, 0.0)
    total = win + draw + loss
    return (win / total, draw / total, loss / total)


def fen_to_graph(fen, wdl):
    """
    Convert a FEN string to a PyG Data object.

    Node features (21d per square):
      - Piece one-hot (12d): WP,WN,WB,WR,WQ,WK,BP,BN,BB,BR,BQ,BK
      - Empty flag (1d)
      - File (1d, normalized), Rank (1d, normalized)
      - Side to move (1d), Castling rights (4d), Half-move clock (1d)

    Edge features (6d per legal move):
      - is_capture, is_promotion, is_castling, is_en_passant, dx, dy
    """
    board = chess.Board(fen)

    side_to_move = 1.0 if board.turn == chess.WHITE else 0.0
    castling = [
        float(board.has_kingside_castling_rights(chess.WHITE)),
        float(board.has_queenside_castling_rights(chess.WHITE)),
        float(board.has_kingside_castling_rights(chess.BLACK)),
        float(board.has_queenside_castling_rights(chess.BLACK)),
    ]
    halfmove = min(board.halfmove_clock, 100) / 100.0
    PIECE_OFFSET = {chess.WHITE: 0, chess.BLACK: 6}

    node_features = []
    for square in chess.SQUARES:
        feat = [0.0] * 21
        piece = board.piece_at(square)
        if piece:
            idx = PIECE_OFFSET[piece.color] + (piece.piece_type - 1)
            feat[idx] = 1.0
        else:
            feat[12] = 1.0
        feat[13] = chess.square_file(square) / 7.0
        feat[14] = chess.square_rank(square) / 7.0
        feat[15] = side_to_move
        feat[16] = castling[0]
        feat[17] = castling[1]
        feat[18] = castling[2]
        feat[19] = castling[3]
        feat[20] = halfmove
        node_features.append(feat)

    src, dst, edge_attrs = [], [], []
    for move in board.legal_moves:
        from_sq, to_sq = move.from_square, move.to_square
        src.append(from_sq)
        dst.append(to_sq)
        dx = (chess.square_file(to_sq) - chess.square_file(from_sq)) / 7.0
        dy = (chess.square_rank(to_sq) - chess.square_rank(from_sq)) / 7.0
        edge_attrs.append([
            float(board.is_capture(move)),
            float(move.promotion is not None),
            float(board.is_castling(move)),
            float(board.is_en_passant(move)),
            dx, dy,
        ])

    if len(src) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, 6), dtype=torch.float)
    else:
        edge_index = torch.tensor([src, dst], dtype=torch.long)
        edge_attr = torch.tensor(edge_attrs, dtype=torch.float)

    x = torch.tensor(node_features, dtype=torch.float)
    y = torch.tensor(wdl, dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)


log.info("Data functions defined.")

In [ ]:
# ============================================================
# Load dataset + stratified sampling + build graphs
# ============================================================

t0 = time.time()
log.info(f"Loading CSV from {DATA_PATH}...")
df = pd.read_csv(DATA_PATH)
log.info(f"  Total rows: {len(df):,}")
log.info(f"  Columns: {list(df.columns)}")

# Parse evaluations to centipawns
df["cp"] = df["Evaluation"].apply(parse_evaluation)
log.info(f"  Eval range: [{df['cp'].min():.0f}, {df['cp'].max():.0f}] cp")
log.info(f"  Eval mean: {df['cp'].mean():.1f}, std: {df['cp'].std():.1f}")

# ── Stratified sampling ──
if NUM_SAMPLES is not None and NUM_SAMPLES < len(df):
    log.info(f"Stratified sampling {NUM_SAMPLES:,} positions...")
    bins = [-float("inf"), -5000, -2000, -1000, -500, -200, -100, -50,
            0, 50, 100, 200, 500, 1000, 2000, 5000, float("inf")]
    df["bin"] = pd.cut(df["cp"], bins=bins, labels=False)

    # Log bin distribution
    bin_counts = df["bin"].value_counts().sort_index()
    log.info(f"  Bin distribution (before sampling):")
    for b, count in bin_counts.items():
        pct = count / len(df) * 100
        target = max(1, int(NUM_SAMPLES * count / len(df)))
        log.info(f"    Bin {b:2d}: {count:>7,} ({pct:5.1f}%) → sample {target:,}")

    sampled = df.groupby("bin", group_keys=False).apply(
        lambda g: g.sample(
            n=min(len(g), max(1, int(NUM_SAMPLES * len(g) / len(df)))),
            random_state=42,
        ),
        include_groups=False,
    )
    sampled = df.loc[sampled.index]

    remaining = NUM_SAMPLES - len(sampled)
    if remaining > 0:
        leftover = df.drop(sampled.index)
        extra = leftover.sample(n=min(remaining, len(leftover)), random_state=42)
        sampled = pd.concat([sampled, extra])

    df = sampled.head(NUM_SAMPLES).reset_index(drop=True)
    if "bin" in df.columns:
        df = df.drop(columns=["bin"])
    log.info(f"  Sampled {len(df):,} positions")
else:
    log.info("Using full dataset (no sampling)")

log.info(f"  CSV loading took {time.time() - t0:.1f}s")

# ── Build graphs (with caching) ──
cache_file = Path(CACHE_PATH) if USE_CACHE else None

if cache_file and cache_file.exists():
    log.info(f"Loading cached graphs from {cache_file}...")
    graphs = torch.load(cache_file, weights_only=False)
    log.info(f"  Loaded {len(graphs):,} cached graphs")
else:
    log.info("Building graphs from FEN positions...")
    t1 = time.time()
    graphs = []
    errors = 0
    for i, row in tqdm(df.iterrows(), total=len(df), desc="FEN → Graph"):
        fen = row["Position"]
        cp = row["cp"]
        wdl = cp_to_wdl(cp)
        try:
            graph = fen_to_graph(fen, wdl)
            graph.cp = torch.tensor([cp], dtype=torch.float)
            graphs.append(graph)
        except Exception as e:
            errors += 1
            if errors <= 5:
                log.warning(f"Error at index {i}, FEN='{fen}': {e}")

    elapsed = time.time() - t1
    log.info(f"  Built {len(graphs):,} graphs in {elapsed:.1f}s ({len(graphs)/elapsed:.0f} graphs/s)")
    if errors > 0:
        log.warning(f"  {errors} errors during graph construction")

    if cache_file:
        torch.save(graphs, cache_file)
        log.info(f"  Cached to {cache_file}")

# ── Quick sanity check ──
g = graphs[0]
log.info(f"Sample graph: {g.x.shape[0]} nodes, {g.edge_index.shape[1]} edges, "
         f"node_dim={g.x.shape[1]}, edge_dim={g.edge_attr.shape[1]}, "
         f"WDL target={[f'{v:.2f}' for v in g.y.tolist()]}")

In [ ]:
# ============================================================
# Train / Val / Test split + DataLoaders
# ============================================================

train_graphs, temp_graphs = train_test_split(graphs, test_size=0.2, random_state=42)
val_graphs, test_graphs = train_test_split(temp_graphs, test_size=0.5, random_state=42)

log.info(f"Split: Train={len(train_graphs):,}  Val={len(val_graphs):,}  Test={len(test_graphs):,}")

train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_graphs, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_graphs, batch_size=BATCH_SIZE)

log.info(f"Batches per epoch: train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}")

# ── Plot WDL distribution ──
train_wdl_classes = [g.y.argmax().item() for g in train_graphs]
class_names = ["Win", "Draw", "Loss"]
counts = [train_wdl_classes.count(i) for i in range(3)]
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(class_names, counts, color=["#2e7d32", "#757575", "#c62828"])
ax.set_ylabel("Count")
ax.set_title("Training Set WDL Class Distribution")
for i, c in enumerate(counts):
    ax.text(i, c + len(train_graphs)*0.01, f"{c:,}\n({c/len(train_graphs)*100:.1f}%)",
            ha="center", fontsize=9)
plt.tight_layout()
plt.show()

## 2. Model Architecture

**ResGATv2**: GATv2Conv with edge features, residual connections, batch normalization.

In [ ]:
# ============================================================
# Model: ResGATv2 for chess WDL prediction
# ============================================================

class ResGATv2Block(nn.Module):
    """Residual block with two GATv2Conv layers."""
    def __init__(self, channels, heads, edge_dim, dropout=0.1):
        super().__init__()
        self.conv1 = GATv2Conv(
            channels, channels // heads, heads=heads,
            edge_dim=edge_dim, concat=True, dropout=dropout,
        )
        self.bn1 = BatchNorm(channels)
        self.conv2 = GATv2Conv(
            channels, channels // heads, heads=heads,
            edge_dim=edge_dim, concat=True, dropout=dropout,
        )
        self.bn2 = BatchNorm(channels)

    def forward(self, x, edge_index, edge_attr):
        residual = x
        x = F.relu(self.bn1(self.conv1(x, edge_index, edge_attr)))
        x = self.bn2(self.conv2(x, edge_index, edge_attr))
        x = F.relu(x + residual)
        return x


class ChessGATv2(nn.Module):
    """Graph Attention Network for chess position WDL evaluation."""
    def __init__(self, node_dim=21, edge_dim=6, hidden=128, heads=4,
                 num_blocks=4, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Sequential(nn.Linear(node_dim, hidden), nn.ReLU())
        self.blocks = nn.ModuleList([
            ResGATv2Block(hidden, heads, edge_dim, dropout=0.1)
            for _ in range(num_blocks)
        ])
        self.head = nn.Sequential(
            nn.Linear(hidden, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 3),  # Win, Draw, Loss
        )

    def forward(self, data):
        x, edge_index, edge_attr, batch = (
            data.x, data.edge_index, data.edge_attr, data.batch
        )
        x = self.input_proj(x)
        for block in self.blocks:
            x = block(x, edge_index, edge_attr)
        x = global_mean_pool(x, batch)
        return self.head(x)


# ── Instantiate ──
model = ChessGATv2(
    hidden=HIDDEN, heads=HEADS, num_blocks=NUM_BLOCKS, dropout=DROPOUT
).to(device)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
log.info(f"Model created: {num_params:,} trainable parameters")
log.info(f"Architecture: {NUM_BLOCKS} ResGATv2 blocks, {HEADS} heads, hidden={HIDDEN}")
print(model)

## 3. Training

In [ ]:
# ============================================================
# Training and evaluation functions
# ============================================================

def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        logits = model(batch)
        target = batch.y.view(-1, 3)
        log_probs = F.log_softmax(logits, dim=1)
        loss = F.kl_div(log_probs, target, reduction="batchmean")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * target.size(0)
        correct += (logits.argmax(1) == target.argmax(1)).sum().item()
        total += target.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_targets, all_cp_true, all_values_pred = [], [], [], []

    for batch in loader:
        batch = batch.to(device)
        logits = model(batch)
        target = batch.y.view(-1, 3)
        log_probs = F.log_softmax(logits, dim=1)
        loss = F.kl_div(log_probs, target, reduction="batchmean")
        total_loss += loss.item() * target.size(0)

        probs = F.softmax(logits, dim=1)
        pred_class = probs.argmax(dim=1)
        true_class = target.argmax(dim=1)
        correct += (pred_class == true_class).sum().item()
        total += target.size(0)

        value_pred = probs[:, 0] - probs[:, 2]
        all_preds.extend(pred_class.cpu().tolist())
        all_targets.extend(true_class.cpu().tolist())
        all_values_pred.extend(value_pred.cpu().tolist())
        if hasattr(batch, "cp"):
            all_cp_true.extend(batch.cp.view(-1).cpu().tolist())

    avg_loss = total_loss / total
    accuracy = correct / total

    # Value MAE and correlation
    values_pred_t = torch.tensor(all_values_pred)
    value_mae, correlation = None, None
    if all_cp_true:
        values_true = torch.tensor([
            2.0 / (1.0 + math.exp(-cp / WDL_K)) - 1.0 for cp in all_cp_true
        ])
        value_mae = (values_pred_t - values_true).abs().mean().item()
        if len(values_true) > 1:
            vp = values_pred_t - values_pred_t.mean()
            vt = values_true - values_true.mean()
            correlation = ((vp * vt).sum() / (vp.norm() * vt.norm() + 1e-8)).item()

    return {
        "loss": avg_loss, "accuracy": accuracy,
        "value_mae": value_mae, "correlation": correlation,
        "preds": all_preds, "targets": all_targets,
        "cp_true": all_cp_true, "values_pred": all_values_pred,
    }


# ── LR schedule: linear warmup + cosine decay ──
def get_scheduler(optimizer, warmup, total):
    def lr_lambda(epoch):
        if epoch < warmup:
            return epoch / max(warmup, 1)
        progress = (epoch - warmup) / max(total - warmup, 1)
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    return LambdaLR(optimizer, lr_lambda)


log.info("Training functions defined.")

In [ ]:
# ============================================================
# Training loop
# ============================================================

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = get_scheduler(optimizer, WARMUP_EPOCHS, EPOCHS)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "val_corr": []}
best_val_loss = float("inf")
patience_counter = 0
train_start = time.time()

log.info(f"Starting training: {EPOCHS} epochs, batch_size={BATCH_SIZE}, lr={LR}, patience={PATIENCE}")
log.info(f"{'Epoch':>5} | {'Train Loss':>10} {'Acc':>6} | {'Val Loss':>10} {'Acc':>6} | {'Corr':>7} {'MAE':>7} | {'LR':>9} | {'Time':>6}")
log.info("-" * 90)

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()

    train_loss, train_acc = train_epoch(model, train_loader, optimizer, device)
    val_result = evaluate(model, val_loader, device)
    scheduler.step()

    lr = optimizer.param_groups[0]["lr"]
    epoch_time = time.time() - epoch_start

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_result["loss"])
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_result["accuracy"])
    history["val_corr"].append(val_result["correlation"])

    corr_s = f"{val_result['correlation']:.4f}" if val_result["correlation"] else "  N/A  "
    mae_s = f"{val_result['value_mae']:.4f}" if val_result["value_mae"] else "  N/A  "

    marker = ""
    if val_result["loss"] < best_val_loss:
        best_val_loss = val_result["loss"]
        patience_counter = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        marker = " *"
    else:
        patience_counter += 1

    log.info(
        f"{epoch:5d} | {train_loss:10.4f} {train_acc:5.1%} | "
        f"{val_result['loss']:10.4f} {val_result['accuracy']:5.1%} | "
        f"{corr_s} {mae_s} | {lr:9.2e} | {epoch_time:5.1f}s{marker}"
    )

    if patience_counter >= PATIENCE:
        log.info(f"Early stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)")
        break

total_time = time.time() - train_start
log.info(f"Training complete in {total_time/60:.1f} minutes")
log.info(f"Best validation loss: {best_val_loss:.4f}")

## 4. Evaluation

In [ ]:
# ============================================================
# Training curves
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(history["train_loss"], label="Train", linewidth=1.5)
axes[0].plot(history["val_loss"], label="Val", linewidth=1.5)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("KL-Div Loss")
axes[0].set_title("Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history["train_acc"], label="Train", linewidth=1.5)
axes[1].plot(history["val_acc"], label="Val", linewidth=1.5)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("WDL Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

valid_corr = [c for c in history["val_corr"] if c is not None]
if valid_corr:
    axes[2].plot(valid_corr, label="Val Correlation", color="green", linewidth=1.5)
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Pearson r")
    axes[2].set_title("Value Correlation")
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()
log.info("Training curves plotted.")

In [ ]:
# ============================================================
# Final test set evaluation
# ============================================================

model.load_state_dict(torch.load(BEST_MODEL_PATH, weights_only=True, map_location=device))
test_result = evaluate(model, test_loader, device)

log.info("=" * 50)
log.info("FINAL TEST SET RESULTS")
log.info("=" * 50)
log.info(f"  Test Loss:       {test_result['loss']:.4f}")
log.info(f"  Test WDL Acc:    {test_result['accuracy']:.1%}")
if test_result["value_mae"] is not None:
    log.info(f"  Value MAE:       {test_result['value_mae']:.4f}")
if test_result["correlation"] is not None:
    log.info(f"  Correlation:     {test_result['correlation']:.4f}")

# ── Per-class accuracy ──
from collections import Counter
class_correct = Counter()
class_total = Counter()
for p, t in zip(test_result["preds"], test_result["targets"]):
    class_total[t] += 1
    if p == t:
        class_correct[t] += 1

for cls_id, cls_name in enumerate(["Win", "Draw", "Loss"]):
    total = class_total.get(cls_id, 0)
    correct = class_correct.get(cls_id, 0)
    acc = correct / total if total > 0 else 0
    log.info(f"  {cls_name:>4s} accuracy: {acc:.1%} ({correct}/{total})")

In [ ]:
# ============================================================
# Confusion matrix
# ============================================================

cm = confusion_matrix(test_result["targets"], test_result["preds"], labels=[0, 1, 2])
disp = ConfusionMatrixDisplay(cm, display_labels=["Win", "Draw", "Loss"])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(cmap="Blues", ax=ax)
ax.set_title("WDL Confusion Matrix (Test Set)")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()
log.info("Confusion matrix plotted.")

In [ ]:
# ============================================================
# Predicted vs true evaluation scatter plot
# ============================================================

if test_result["cp_true"]:
    cp_pred = []
    for v in test_result["values_pred"]:
        v_clamped = max(-0.999, min(0.999, v))
        p_win = (v_clamped + 1.0) / 2.0
        p_loss = 1.0 - p_win
        if p_loss < 1e-6:
            cp_pred.append(5000)
        elif p_win < 1e-6:
            cp_pred.append(-5000)
        else:
            cp_pred.append(WDL_K * math.log(p_win / p_loss))

    cp_true_clamped = [max(-5000, min(5000, c)) for c in test_result["cp_true"]]

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.scatter(cp_true_clamped, cp_pred, alpha=0.05, s=2)
    ax.plot([-5000, 5000], [-5000, 5000], "r--", linewidth=1, label="Perfect")
    ax.set_xlabel("True Evaluation (cp)")
    ax.set_ylabel("Predicted Evaluation (cp)")
    ax.set_title("Predicted vs True Evaluation")
    ax.legend()
    ax.set_xlim(-5000, 5000)
    ax.set_ylim(-5000, 5000)
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.savefig("eval_scatter.png", dpi=150)
    plt.show()
    log.info("Scatter plot generated.")
else:
    log.warning("No centipawn data available for scatter plot.")

In [ ]:
# ============================================================
# Sample predictions — inspect a few positions
# ============================================================

log.info("Sample predictions from test set:")
log.info(f"{'FEN (truncated)':>45s} | {'True CP':>8s} | {'True WDL':>12s} | {'Pred WDL':>12s} | {'Pred Val':>8s}")
log.info("-" * 100)

sample_indices = np.random.RandomState(42).choice(len(test_graphs), size=min(10, len(test_graphs)), replace=False)
for idx in sample_indices:
    g = test_graphs[idx]
    batch = Batch.from_data_list([g]).to(device)
    with torch.no_grad():
        logits = model(batch)
        probs = F.softmax(logits, dim=1)[0]

    cp = g.cp.item()
    true_wdl = g.y.tolist()
    pred_wdl = probs.cpu().tolist()
    pred_val = pred_wdl[0] - pred_wdl[2]

    # Reconstruct FEN from the graph's context (stored during build)
    fen_short = f"cp={cp:.0f}"

    true_str = f"W:{true_wdl[0]:.2f} D:{true_wdl[1]:.2f} L:{true_wdl[2]:.2f}"
    pred_str = f"W:{pred_wdl[0]:.2f} D:{pred_wdl[1]:.2f} L:{pred_wdl[2]:.2f}"

    log.info(f"{fen_short:>45s} | {cp:>8.0f} | {true_str:>12s} | {pred_str:>12s} | {pred_val:>+8.3f}")

In [ ]:
# ============================================================
# Save model for download
# ============================================================

log.info(f"Best model saved at: {BEST_MODEL_PATH}")
log.info(f"Model size: {Path(BEST_MODEL_PATH).stat().st_size / 1024:.1f} KB")
log.info("")
log.info("To use this model locally for playing:")
log.info("  1. Download best_model.pt")
log.info("  2. Place it in your ChessGCN project folder")
log.info("  3. Run: python play.py --checkpoint best_model.pt")